# Shortest Path

In [ ]:
from __future__ import annotations

import os

import h5py
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from graph_prep import sampling, short_path, transforms

In [ ]:
main_dir = os.path.dirname(os.getcwd())
forward_link_path = os.path.join(main_dir, "data/output/enwiki_forward_links.hdf5")
matrix_info_path = os.path.join(main_dir, "data/output/enwiki_matrix_index_info.tsv")

In [ ]:
with h5py.File(forward_link_path, "r") as forw_f:
    forward_locs = forw_f["forward_locs"][:]
    forward_vals = forw_f["forward_vals"][:]

In [ ]:
backward_locs, backward_vals = transforms.swap_graph_direction(
    forward_locs, forward_vals, verbose=True
)

In [ ]:
matrix_vals_df = pd.read_csv(matrix_info_path, sep="\t")

### Get a couple example paths

In [ ]:
path_dest_list = [[1, 1125003], [12345, 543214], [12243, 78778], [10, 500]]

path_list = []

for path_dest in path_dest_list:
    path_list.append(
        short_path.path_one_way(
            path_dest[0],
            path_dest[1],
            forward_locs,
            forward_vals,
            backward_locs,
            backward_vals,
        )
    )

for path in path_list:
    for page in path[:-1]:
        print(matrix_vals_df.iloc[page - 1]["PageTitle"], end=" -> ")

    print(matrix_vals_df.iloc[path[-1] - 1]["PageTitle"])

### Path Histograms

In [ ]:
def plot_graph(steps_list: list[int], language: str, max_steps=10) -> None:
    plt.figure(dpi=200)
    plt.hist(
        steps_list,
        bins=np.arange(0, max_steps + 1, 1) - 0.5,
        rwidth=0.8,
        density=True,
        zorder=10,
        color="0.3",
    )

    plt.xlabel("Number of Steps")
    plt.ylabel("Fraction of Pairs of Pages")
    plt.title(f"Steps for Shortest Path {language} Wikipedia")

    plt.xticks(
        np.arange(0, max_steps, 1),
        labels=["NOT\nFOUND", *list(np.arange(1, max_steps, 1).astype(str))],
    )

    plt.grid()

    plt.show()

In [ ]:
en_steps_list = sampling.generate_steps_list(
    100, 10, forward_locs, forward_vals, backward_locs, backward_vals
)

In [ ]:
plot_graph(en_steps_list, "English")

In [ ]:
zh_forward_path = os.path.join(main_dir, "data/output/zhwiki_forward_links.hdf5")
fr_forward_path = os.path.join(main_dir, "data/output/zhwiki_forward_links.hdf5")

In [ ]:
with h5py.File(zh_forward_path, "r") as forw_f:
    zh_forward_locs = forw_f["forward_locs"][:]
    zh_forward_vals = forw_f["forward_vals"][:]

In [ ]:
zh_backward_locs, zh_backward_vals = transforms.swap_graph_direction(
    zh_forward_locs, zh_forward_vals, verbose=True
)

In [ ]:
zh_steps_list = sampling.generate_steps_list(
    100, 10, zh_forward_locs, zh_forward_vals, zh_backward_locs, zh_backward_vals
)

In [ ]:
plot_graph(zh_steps_list, "Chinese")

In [ ]:
with h5py.File(fr_forward_path, "r") as forw_f:
    fr_forward_locs = forw_f["forward_locs"][:]
    fr_forward_vals = forw_f["forward_vals"][:]

In [ ]:
fr_backward_locs, fr_backward_vals = transforms.swap_graph_direction(
    fr_forward_locs, fr_forward_vals, verbose=True
)

In [ ]:
fr_steps_list = sampling.generate_steps_list(
    100, 10, fr_forward_locs, fr_forward_vals, fr_backward_locs, fr_backward_vals
)

In [ ]:
plot_graph(fr_steps_list, "French")